In [17]:
# imports

import os
from dotenv import load_dotenv
import requests
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import anthropic

In [25]:
load_dotenv(override=True)

True

In [26]:
# Sign in to HuggingFace Hub

hf_token = os.getenv("HF_TOKEN")
deepgram_token = os.getenv("DEEPGRAM_API_KEY")

login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
# Audio

audio_file = "data/testaudio.mp3"

if not deepgram_token:
    raise ValueError("DEEPGRAM_API_KEY is not set")

with open(audio_file, "rb") as audio:
    response = requests.post(
        "https://api.deepgram.com/v1/listen",
        params={
            "model": "nova-3",
            "smart_format": "true",
            "punctuate": "true",
            "diarize": "true",
        },
        headers={
            "Authorization": f"Token {deepgram_token}",
            "Content-Type": "audio/mp3", 
        },
        data=audio,
    )

response.raise_for_status()
transcription_result = response.json()

transcript = transcription_result["results"]["channels"][0]["alternatives"][0]["transcript"]
print(transcript)

Hi, I'm Ria. Thank you for coming to interview with us today. Let's just go ahead and get started with a quick introduction. Could you tell me a little bit about yourself? Yeah, absolutely. First of all, thank you so much for having me. It's an honor to be here. So a bit about myself, I'm originally from Morocco, where I lived pretty much my whole life. I then came to Boston and studied industrial engineering at Northeastern University. While I was in college, did a few different internships. I did technical ones at a local power plant and at Amazon Robotics, and I also did one in consulting at the Boston Consulting Group, and was also the president of the Consulting Club at my university. So really I'm someone who enjoys both the business side of things, but I also do have a passion for technology. So down the line, my goal is to start my own education technology company in my home country Morocco, so I can make education more affordable through the use of tech. And outside of school 

In [27]:
# Model

claude = anthropic.Anthropic()

In [28]:
# Prompts

name = "Ziyah"
role = "Consultant"
company = 'Accenture'
interviewer = '(not provided)'

system_prompt = """
You are a helpful assistant that takes transcripts of interviews and produces helpful insights
and opportunities for future improvement for the usesr, in markdown format without code blocks.
"""

user_prompt = f"""
Below is the transcript of {name}'s interview with {interviewer} from {company}.
Please provide a summary of the interview and questions asked, insights into
how the interview went, and opportunities for future improvement. Base your insights on
the transcript provided, and additional research about the role, company, and interviewer if provided.
The summary should be in markdown format without code blocks."



Transcription:
{transcript}
"""

In [ ]:
response = claude.messages.create(
    model='claude-haiku-4-5',
    max_tokens=10000,
    system=system_prompt,
    messages=[{"role": "user", "content": user_prompt}]
    )

reply = response.content[0].text

In [32]:
display(Markdown(reply))

# Interview Summary and Analysis

## Overview
Ziyah interviewed with Ria at Accenture for a consulting position. The candidate demonstrated strong technical and business experience with a clear career vision, presenting himself as poised, well-prepared, and genuinely interested in the firm.

## Questions Asked

1. **Introduction** - Tell me about yourself
2. **Motivation** - Why are you interested in interviewing with Accenture?
3. **Problem-Solving** - Tell me about a time you solved a difficult problem
4. **Strengths (Primary)** - What are your key strengths?
5. **Strengths (Deep Dive)** - Tell me about a time you persuaded someone to change their mind
6. **Strengths (Second)** - Tell me about a time you demonstrated strong leadership
7. **Weaknesses** - Tell me about a weakness you face in